# Redundancy Detection with Dleto Chiseling

`CC-BY 2025 Brooksbank, Kassabov, Wilson`

This tutorial uses Dleto's "Tucker Chisels" to reproduce a Tucker Decompositions, also known as radical/total zero-divisor detection.

> A **Tucker Decomposition** of a tensor $\Gamma$ framed by axes (a.k.a. modes/legs/indices) $\mathbb{K}^{d_1},\ldots, \mathbb{K}^{d_{\ell}}$, is a subset of axes $A\subset\{1,\ldots,\ell\}$ and a decomposition
> $$\forall a\in A,\qquad \mathbb{K}^{d_a}=E_a\oplus R_a$$
> such that $\Gamma$ contracted on $R_a$ is 0.  

The decompositions can be given by a partitioned bases of $E_a$ and $R_a$, or equivalently by a linear projections $e_a:\mathbb{K}^{d_a}\to \mathbb{K}^{e_a}$ with kernel $R_a$.  In many situations the purpose of a Tucker decomposition is to restrict the tensor the $E_a$-spaces.  In that case it is sufficient to return only $E_a$ (respectively $e_a$) and to refer to that restriction as the Tucker Decomposition.  `Dleto` offers both forms of Tucker decomposition as an option.

 1. [Loading Dleto](#1-loading-dleto)
 2. [Creating a Tensor Experiment](#2-creating-tensor-experiment)
 3. [Tucker chiseling](#3-tucker-chiseling)
 4. Derivation selection.
___

**Performance Remark.** Tucker decompositions have been explored since the 1800's and as such there are many optimized strategies to discover them.  This tutorial therefore should be seen as using a familiar problem to explore the range of options of Dleto chisels and the use of the parameters available to Dleto chiseling.  Unfortunately, Dleto chiselling operates with a complexity slightly greater than many more direct strategies for Tucker decompositions.  For high-performance computations, Dleto automatically switches to alternative optimized strategies by calling `nondeg`. 


## 1. Loading Dleto

Start by loading `Dleto.jl`.  If this is your first time you may need to install auxiliary packages and possibly set up Julia for notebooks.  That is a one-time setup for most users, see instructions here or consider using the fully online Binder demonstration.

In [ ]:
# Uncomment and run the first time, if Dleto is not installed
# using Pkg

# Option 1: To install from remote repository, use:
# Pkg.add(url="https://github.com/thetensor-space/OpenDleto")

# Option 2: If cloned locally at PATH 
# Pkg.activate( PATH ) 

If you have already added Dleto to your Julia packages begin by loading the necessary packages, `ITensors` for general tensor controls, `Plots` for visualization tools, and `Dleto` the primary package of chisel techniques.

In [ ]:
using ITensors
using Plots
using Dleto

## 2 Creating Tensor Experiment

Our first experiment is the simplest demonstration of chiseling uncovering some form of structure.  

We create two tensors, one randomized for control, and an identical in size but with 2 rows, columns, and slices set to all zero.  We then randomize the experiment tensor by applying a change in coordinates.  The goal is to using chiseling to detect and recover the hidden zero rows/columns/slices.

We note that this experiment is extremely basic and there are faster algorithms to recover this structure known under the names of **detecting radicals** or **Tucker decompositions**.  `Delto.jl` makes use of those faster methods for large scale experiments, but this experiment simply uses generic Delto methods to explore what is possible.


> **Note** Julia being mathematically oriented accepts $\LaTeX$ styled commands with tab-completion.  For example to insert the Unicode character for $\Gamma$ use `\Gamma` in the code area followed by `tab` (or cut-and-paste a character you see somewhere else).  Or replace with an simpler string of characters of your liking.  One suggestion, `Dleto.jl` calculations make substantial use of tensors, matrices, and lists of matrices.  A convention that clearly indicates those roles will be an investment worth your time.  
>
> We will be using:
> * capitol Greek letters `Γ` (`\Gamma`), `Δ` (`\Delta`), `Σ` (`\Sigma`), `Ξ` (`\Xi`), `Υ` (`\Upsilon`), etc. for tensors
> * capital English letters `X`, `Y`, `Z` etc. for matrices
> * lower case Geek letters for real numbers
> * lower case English letters for integers
> * Plural for lists of data, for instance `Γs` and `Xs`.

In [ ]:
ds = (5,5,5); rs=(3,3,3)
Γ = randn(Float64, ds);  # a control tensor

# the experiment tensor is initially 0 but we fill in a subregion.
Δ = zeros(Float64, ds);  
Δ[1:rs[1], 1:rs[2], 1:rs[3]] .= randn(rs...)

side_by_side(Γ, Δ; left_title="Control Γ", right_title="Experiment Δ")

For larger tensors it may help to do a visualization instead, and we can do this with the following.

In [ ]:
p1 = plot_tensor(Γ; title="Control Γ", color=:blue)
p2 = plot_tensor(Δ; title="Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

The control tensor very likely has few to no all zero axes whereas the experiment evidently does.  The role of chiseling is to recover such anamolies without knowing ahead of time.  So we now randomize the experiment so as to obscure this data.  Note that the randomization applies random bases change so the original coordinates.  

It should now be much less obvious that the control and experiment are any different.

In [ ]:
Γ_rand, X1s = randomize_tensor(Γ)
@assert isapprox(Γ * X1s, Γ_rand)

Δ_rand, Y1s = randomize_tensor(Δ)
@assert isapprox(Δ * Y1s, Δ_rand)

You may discover `randomize_tensor` has switched the internal representation of tensors the `ITensor` format.  In fact this why we are able to apply a **list** (Julia `Vector` types) of matrices to `Δ` without specifying what side to multiply on.  The order is immaterial, for example `Xs*Δ` is allowable as well.  This is in contrast to matrix multiplication where sides indicate the axis of application.  `ITensors` uses internal markers to permit the correct application across multiple axes.

To convert back to arrays you may use `Array(data, inds(data))` but `ITensors` format offers multiple convenient features for tensors and organizes the information to avoid common errors so the ideal situation is to adjust to the use of `ITensors`.

In [ ]:
side_by_side(Γ_rand, Δ_rand;
            left_title="Control Γ", right_title="Experiment Δ_rand")

Or see it again as a plot.

In [ ]:
p1 = plot_tensor(Γ_rand; title="Randomized Control Γ", color=:blue)
p2 = plot_tensor(Δ_rand; title="Randomized Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

The goal now is to detect the rows, columns, and slices of 0's, that is the Tucker decomposition, of the experiment.  As warned, there is a more efficient tool than chiseling and just to help our users be aware to use the right tool here it is in practice before we continue with our chiselling experiment anyway.

In [ ]:
Γ_nondeg, X2s = Dleto.nondeg(Γ_rand, mode=:full);  # use mode=:trunc to truncate the 0's in the Tucker decomposition
@assert isapprox(Γ_rand * X2s, Γ_nondeg)
Δ_nondeg, Y2s = Dleto.nondeg(Δ_rand, mode=:full);
@assert isapprox(Δ_rand * Y2s, Δ_nondeg)

p1 = plot_tensor(Γ_nondeg; title="Tucker Control Γ", color=:blue)
p2 = plot_tensor(Δ_nondeg; title="Tucker Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

It is helpful to not that Tucker decompositions are a closure, once applied, a second application achieves no new clusters (though it may reorder and rescale the one already identifies).  

In [ ]:
Δ_nondeg_trunc, _ = nondeg(Δ_rand, mode=:trunc);
Δ_nondeg2_trunc, _ = nondeg(Δ_nondeg_trunc, mode=:trunc);
@assert size(Δ_nondeg_trunc) == size(Δ_nondeg2_trunc)

# 3. Tucker Chiseling

While the command `nondeg` performs the Tucker decomposition on one step, it does so with internal optimizations that mostly do not relate to the chisel strategy more generally.  So to break-down the Tucker decomposition using Tucker Chisels specifically we make a 3 step process.
 * Select the appropriate Tucker chisel
 * Compute derivations of the tensor with the Tucker chisel.
 * Use the derivations to stratify the original tensor.

First let us use the default universal chisel to see what happens.

In [ ]:
Γ_strat, X3s = stratify(Γ_rand);
Δ_strat, Y3s = stratify(Δ_rand);

Take note of the different number of derivations discovered.  This hints at how Dleto identifies the hidden structure.  We will return to this at the end of this section.

Now let us plot the results of generic stratification.  In a typical situation the control tensor will remain random scatter plot of values spread across all axes.  Meanwhile the stratified experiment will recover a cluster surrounded by nearly zero values.  The precise position of the nonzeros can vary but the total nonzero cluster should be of dimensions `ds-rs`.

In [ ]:
p1 = plot_tensor(Γ_strat; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_strat; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

The plots here are showing dots whose volume is proportional to scalar size, which can omit tiny values.  Inspecting the actual data tells the store in more detail.  In particular we do not manage to up a small tolerance we see that we have recovered the degeneracy planted in the experimental tensor.

In [ ]:
side_by_side(Γ_strat, Δ_strat;
            left_title="Stratified Control Γ", right_title="Stratified Experiment Δ")

If we return to the starting cell of this section we may observe that the stratification output for the control printed the following clues.
```
Found 2 derivations for stratification.
Found 10 derivations for stratification.
```
This is a hint that the underlying algebraic structure of these tensors is considerably different.  In fact the value of 10 for the experiment is merely the first cut-off and it is likely that this tensor could have many more derivations for stratification.  To explore this we can intervene in the default parameters and ask for all the derivations before stratification.

In [ ]:
Γ_der = der(Γ_rand; nd=-1);  # nd=number of derivations, if negative find all.
Δ_der = der(Δ_rand; nd=-1);
println("Control Γ has $(length(Γ_der)) derived tensors.")
println("Experiment Δ has $(length(Δ_der)) derived tensors.")

If you inspect the size of the derivations our experiment has on the order of at least 
    $$2+ds[1]*(ds[1]-rs[1])+ds[2]*(ds[2]-rs[2])+ds[3]*(ds-rs[3])$$ 
derivations ($2+5\cdot 2+5\cdot 2+5\cdot 2=32$ for the default parameters).  The origin of this formula can be loosely explained as the number of bases of each axis that when applied to the tensor contract to 0.  The additional plus $2$ correspond to scalar transforms which do not produce a proper.  Thus when a derivation computation of a tensor returns with only scalar derivations the effect is to detect nothing.

# 3. Larger scale

Now let us increase the scale.  

**Warning.** While playing with parameters it is tempting to introduce ever larger values.  Keep in mind that the total parameters of a tensor grow as the **product of the dimensions**.  Thus to go from (5,5,5) to (25,25,25) is to increase by 125 times, and (50,50,50) is 1000 times more. 

In [ ]:
ds = (25,25,25); rs=(20,22,19)
Γ = randn(Float64, ds);  # a control tensor
Γ_rand, Xs = randomize_tensor(Γ);

# the experiment tensor is initially 0 but we fill in a subregion.
Δ = zeros(Float64, ds);  
Δ[1:rs[1], 1:rs[2], 1:rs[3]] .= randn(rs...)
Δ_rand, Ys = randomize_tensor(Δ);  # In theory Δ_rand = Δ*Xs

# Plotted these two tensors are essentially indistinguishable from each other.
p1 = plot_tensor(Γ_rand; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_rand; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

As a reference point let us search out the Tucker decomposition in the more efficient manner and plot the results.  Make sure to observe the different range in results.

In [ ]:
@time Γ_nondeg, Xs = Dleto.nondeg(Γ_rand, mode=:full);  # use mode=:trunc to truncate the 0's in the Tucker decomposition
@assert isapprox(Γ_rand * Xs, Γ_nondeg)
@time Δ_nondeg, Ys = Dleto.nondeg(Δ_rand, mode=:full);
@assert isapprox(Δ_rand * Ys, Δ_nondeg)

In [ ]:
p1 = plot_tensor(Γ_nondeg; title="Tucker Control Γ", color=:blue)
p2 = plot_tensor(Δ_nondeg; title="Tucker Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

Now lets us see what happens when we attempt to stratify these two tensors in their randomized form.  In this range we also begin to experience warnings brought on by possible issues with numerical approximation.

In [ ]:
Γ_strat, Xs = stratify(Γ_rand);
Δ_strat, Ys = stratify(Δ_rand);

If we plot the tensors we see the results capture clustering.  

In [ ]:

p1 = plot_tensor(Γ_strat; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_strat; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

# 4. Tucker Chisels

So far we have stratified with the default **unviersal chisel** which theoretically capable to detecting aspects of all structures that can be sculpted from a chisel.  This is why in the experiment worked.  However in some special situations we may have a mix of structures, such decompostion of a Tucker type with an unrelated stratification.   In this case changing the chisel can clear out the information.


In [ ]:
tc = TuckerChisel(3); # the number of axes/modes is 3
@time ders = der(tc, Δ_rand; nd=-1);
println("Tucker derivations found: ", length(ders))

Recall our earlier formula of `2+ds[1]*rs[1]+ds[2]*rs[2]+ds[3]*rs[3]` for the number of expected derivations.

In [ ]:
println("Expected number of universal derivations: ", 2+sum(a-> ds[a]*(ds[a]-rs[a]), 1:3))

So we find that the Tucker derivations have focus on just the derivations which related to the bases that could contract with the tensor to 0.  The scalar derivations (and any other derivations) are no excluded from the search by having selected the Tucker chisel.

In [ ]:
@time Δ_strat_2, _ = stratify(Δ_rand, ders[200]);
p = plot_tensor(Δ_strat_2; title="Stratified Experiment Δ from derivation", color=:red)
Plots.plot(p)

In [ ]:
tc = TuckerChisel([true,false,true]); # the number of axes/modes is 3
@time ders = der(tc, Δ_rand; nd=-1);
println("Tucker {1,3}-derivations found", size(ders))